In [4]:
import json
import pandas as pd
from pathlib import Path

In [5]:

def load_recipe_data(file_path):
    """
    Load and process recipe JSON data

    """
    # Read JSON file
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Display basic information about the dataset
    print(f"\nDataset Info for {Path(file_path).name}:")
    print(f"Number of recipes: {len(df)}")
    if 'ingredients' in df.columns:
        print(f"Total unique ingredients: {len(set([ing for ingredients in df['ingredients'] for ing in ingredients]))}")
    
    # Display sample entries
    print("\nSample Recipe:")
    print(df.iloc[0].to_dict())
    
    return df

In [6]:
train_path = r'C:\Users\anujn\OneDrive\Documents\Capstone\ing\train.json'
test_path = r'C:\Users\anujn\OneDrive\Documents\Capstone\ing\test.json'

In [7]:
train_df = load_recipe_data(train_path)
test_df = load_recipe_data(test_path)


Dataset Info for train.json:
Number of recipes: 39774
Total unique ingredients: 6714

Sample Recipe:
{'id': 10259, 'cuisine': 'greek', 'ingredients': ['romaine lettuce', 'black olives', 'grape tomatoes', 'garlic', 'pepper', 'purple onion', 'seasoning', 'garbanzo beans', 'feta cheese crumbles']}

Dataset Info for test.json:
Number of recipes: 9944
Total unique ingredients: 4484

Sample Recipe:
{'id': 18009, 'ingredients': ['baking powder', 'eggs', 'all-purpose flour', 'raisins', 'milk', 'white sugar']}


In [5]:
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
import pandas as pd
import numpy as np

class EnhancedIngredientSubstitution:
    def __init__(self, train_df):
        self.train_df = train_df
        self.ingredient_contexts = defaultdict(list)
        self.cuisine_ingredients = defaultdict(set)
        self.ingredient_frequencies = defaultdict(int)
        self.ingredient_cuisines = defaultdict(set)
        self.ingredient_pairings = defaultdict(lambda: defaultdict(int))
        self.vectorizer = TfidfVectorizer()
        
        self._build_knowledge_base()
        self._create_ingredient_embeddings()
        
    def _build_knowledge_base(self):
        """Build comprehensive knowledge base of ingredients"""
        for _, row in self.train_df.iterrows():
            ingredients = row['ingredients']
            cuisine = row.get('cuisine', 'unknown')
            
            # Update cuisine mappings
            self.cuisine_ingredients[cuisine].update(ingredients)
            
            # Update ingredient frequencies and cuisines
            for ingredient in ingredients:
                self.ingredient_frequencies[ingredient] += 1
                self.ingredient_cuisines[ingredient].add(cuisine)
                
                # Update ingredient contexts
                context = [ing for ing in ingredients if ing != ingredient]
                self.ingredient_contexts[ingredient].extend(context)
                
                # Update ingredient pairings
                for other_ing in context:
                    self.ingredient_pairings[ingredient][other_ing] += 1
    
    def _create_ingredient_embeddings(self):
        """Create ingredient embeddings based on their context"""
        ingredient_docs = {
            ing: ' '.join(contexts) 
            for ing, contexts in self.ingredient_contexts.items()
        }
        
        self.ingredients = list(ingredient_docs.keys())
        self.context_matrix = self.vectorizer.fit_transform(
            [ingredient_docs[ing] for ing in self.ingredients]
        )
    
    def find_substitutes(self, ingredient, cuisine=None, n=1):
        """Find substitute ingredients"""
        if ingredient not in self.ingredients:
            return None
            
        ing_idx = self.ingredients.index(ingredient)
        similarities = cosine_similarity(
            self.context_matrix[ing_idx:ing_idx+1], 
            self.context_matrix
        ).flatten()
        
        sorted_indices = np.argsort(similarities)[::-1]
        
        for idx in sorted_indices:
            candidate = self.ingredients[idx]
            if candidate != ingredient:
                if cuisine and candidate not in self.cuisine_ingredients[cuisine]:
                    continue
                return candidate  # Return the most similar substitute

        return None  # No suitable substitute found

    def save_ingredient_substitutions(self, output_path="Ingredient_Substitutes.csv"):
        """Save each unique ingredient, its substitute, and cuisine type to a CSV file"""
        data = []
        for ingredient in self.ingredients:
            cuisines = list(self.ingredient_cuisines[ingredient])
            substitute = self.find_substitutes(ingredient)
            if substitute:
                for cuisine in cuisines:
                    data.append([ingredient, substitute, cuisine])
        
        df = pd.DataFrame(data, columns=["Ingredient", "Substitute", "Cuisine"])
        df.to_csv(output_path, index=False)

def get_detailed_substitutions(train_df, ingredient):
    system = EnhancedIngredientSubstitution(train_df)
    return system.get_detailed_recommendations(ingredient)

def analyze_recipe(train_df, ingredients):
    system = EnhancedIngredientSubstitution(train_df)
    return system.analyze_recipe_substitutions(ingredients)

# Example usage
system = EnhancedIngredientSubstitution(train_df)
system.save_ingredient_substitutions("Ingredient_Substitutes.csv")


In [7]:
!pip install neo4j

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from neo4j import GraphDatabase

# Replace these with your Neo4j AuraDB credentials
NEO4J_URI = "neo4j+s://69c22d86.databases.neo4j.io"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "wGkq9mqjmS7jqf_ELjnro0ZOdD8KRFFP9dHYZvsxonk"

# Connect to Neo4j AuraDB
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
def test_connection(tx):
    result = tx.run("RETURN 'Connected to Neo4j AuraDB' AS message")
    for record in result:
        print(record["message"])

with driver.session() as session:
    session.read_transaction(test_connection)

# Close connection
driver.close()


C:\Users\anujn\AppData\Local\Temp\ipykernel_19260\3015295703.py:18: DeprecationWarning: read_transaction has been renamed to execute_read
  session.read_transaction(test_connection)


Connected to Neo4j AuraDB


In [ ]:
import pandas as pd

# Load dataset from Kaggle
file_path = r"C:\Users\anujn\OneDrive\Documents\Capstone\ing\Ingredient_Substitutes.csv"
df = pd.read_csv(file_path)

In [ ]:
# Function to store ingredient substitutions in Neo4j AuraDB
def store_ingredients_in_neo4j(tx, ingredient, substitute, cuisine):
    query = """
    MERGE (i:Ingredient {name: $ingredient})
    MERGE (s:Ingredient {name: $substitute})
    MERGE (i)-[:SUBSTITUTES_IN {cuisine: $cuisine}]->(s)
    """
    tx.run(query, ingredient=ingredient, substitute=substitute, cuisine=cuisine)

# Store data in Neo4j
with driver.session() as session:
    for _, row in df.iterrows():
        ingredient = row["Ingredient"].strip()
        substitute = row["Substitute"].strip()
        cuisine = row["Cuisine"].strip()
        session.write_transaction(store_ingredients_in_neo4j, ingredient, substitute, cuisine)

print("Ingredient substitutions successfully stored in Neo4j AuraDB!")

# Close connection
driver.close()

<ipython-input-12-d9087b6fc7bd>:11: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
<ipython-input-12-d9087b6fc7bd>:16: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(store_ingredients_in_neo4j, ingredient, substitute, cuisine)


In [7]:
import random
from neo4j import GraphDatabase

# Class to connect to Neo4j and retrieve ingredient substitutes
class IngredientSubstitution:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def find_substitutes(self, ingredient, cuisine=None):
        with self.driver.session() as session:
            # Query to fetch substitutes (with optional cuisine filtering)
            query = """
            MATCH (i:Ingredient {name: $ingredient})-[r:SUBSTITUTES_IN]->(s:Ingredient)
            WHERE $cuisine IS NULL OR r.cuisine = $cuisine
            RETURN s.name AS substitute, r.cuisine AS cuisine
            """
            result = session.run(query, ingredient=ingredient, cuisine=cuisine)
            substitutes = [(record["substitute"], record["cuisine"]) for record in result]
            
            return substitutes

# Connection details (update with your credentials)
URI = "neo4j+s://6ed640e2.databases.neo4j.io"
USER = "neo4j"
PASSWORD = "LfgFEw52VvIgdie-iqIwbGjpIZJEiEJ9qOos3jydkNI"

# Initialize Neo4j connection
neo4j_conn = IngredientSubstitution(URI, USER, PASSWORD)

# User input for ingredient search
ingredient = input("Enter an ingredient to find substitutes: ").strip().lower()
cuisine = input("Do you want substitutes from a specific cuisine? (Press Enter to skip): ").strip().lower() or None

# Fetch substitutes
substitutes = neo4j_conn.find_substitutes(ingredient, cuisine)

# Output results
if substitutes:
    if len(substitutes) == 1:
        substitute, cuisine_type = substitutes[0]
        if cuisine:  
            print(f"Substitute for '{ingredient}': {substitute} (Cuisine: {cuisine_type})")  
        else:  
            print(f"Substitute for '{ingredient}': {substitute}")
    elif cuisine:
        print(f"Substitutes for '{ingredient}' in {cuisine} cuisine:")
        for substitute, _ in substitutes:
            print(f"- {substitute}")
    else:
        # If no cuisine is specified and multiple substitutes exist, pick one randomly
        substitute, cuisine_type = random.choice(substitutes)
        if cuisine:  
            print(f"Suggested substitute for '{ingredient}': {substitute} (Cuisine: {cuisine_type})")  
        else:  
            print(f"Suggested substitute for '{ingredient}': {substitute}")
else:
    print("No substitutes found.")

# Close connection
neo4j_conn.close()

Substitute for 'salt': kosher salt (Cuisine: mexican)
